# M2: Sampling

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from plotly import graph_objects as go
from plotly.subplots import make_subplots

## a) Signal erzeugen

10000 Samples über 5 s ergeben $f_s = 10000 / 5\,\mathrm{s} = 2000\,\mathrm{Hz}$, wenn der Zeitvektor wie hier mit `np.arange` erzeugt wird.
Mit `np.linspace(0, 5, 10000)` liegen beide Enden im Signal, es gibt nur 9999 Abstände und damit $f_s = 1999.8\,\mathrm{Hz}$ (so in der Matlab-Lösung).

In [ ]:
sampling_frequency = 2e3
start_time, stop_time = 0, 5
time = np.arange(start_time, stop_time, 1 / sampling_frequency)
print(f"{time.size=}, Abtastfrequenz: {1 / (time[1] - time[0]):.1f} Hz")

amplitudes = (1.0, 1.2, 0.6)
frequencies = (1.0, 12.0, 45.0)
phases = (0.0, 3 * np.pi, np.pi / 2)  # cos(x) = sin(x + pi/2)
components = [
    a * np.sin(2 * np.pi * f * time + p)
    for a, f, p in zip(amplitudes, frequencies, phases, strict=True)
]
signal = np.sum(components, axis=0)

# Teilsignale per Klick auf die Legende einblenden
fig = go.Figure(go.Scatter(x=time, y=signal, name="Signal"))
for f, component in zip(frequencies, components, strict=True):
    fig.add_trace(
        go.Scatter(x=time, y=component, name=f"{f:g} Hz", visible="legendonly")
    )
fig.update_layout(
    xaxis=dict(title="Zeit", ticksuffix=" s", range=[0, 1]),
    yaxis=dict(title="Amplitude"),
)
fig.show()

## b) `sample`

Bei $f_\mathrm{sample} = f_\mathrm{signal} / n$ bleibt jeder $n$-te Wert übrig.
Die Zeitachse wird mit derselben Funktion abgetastet, damit die Punkte an den echten Abtastzeitpunkten liegen.

In [ ]:
subsampling_factor = 10
subsampling_frequency = sampling_frequency / subsampling_factor
print(f"{sampling_frequency=}, {subsampling_frequency=}")

In [ ]:
def sample(signal, sampling_frequency, new_sampling_frequency):
    if new_sampling_frequency > sampling_frequency:
        raise ValueError("The new sampling frequency can't be larger than the old one!")

    # round statt //: 2000 // (2000 / 7) ergibt 6.0 (Gleitkomma)
    sampling_step = round(sampling_frequency / new_sampling_frequency)
    # python: slicing:  start:stop:step
    # matlab: indexing: start:step:stop
    return signal[::sampling_step]


sampled_signal = sample(signal, sampling_frequency, subsampling_frequency)
sampled_time = sample(time, sampling_frequency, subsampling_frequency)

fig = go.Figure()
fig.add_trace(
    go.Scatter(x=sampled_time, y=sampled_signal, name="Sampled", mode="markers")
)
fig.add_trace(
    go.Scatter(
        x=time, y=signal, name="Original", line=dict(color="rgba(255, 0, 0, 0.25)")
    )
)
fig.update_layout(
    xaxis=dict(title="Zeit", ticksuffix=" s", range=[0, 0.5]),
    yaxis=dict(title="Amplitude"),
)
fig.show()

## c) `sample_and_hold`

Jeder Abtastwert wird $n$-mal wiederholt. Das Ergebnis hat wieder die Länge und Zeitachse des Originals.

In [ ]:
def sample_and_hold(signal, sampling_frequency, new_sampling_frequency):
    sampling_step = round(sampling_frequency / new_sampling_frequency)
    sampled = sample(signal, sampling_frequency, new_sampling_frequency)
    # falls die Länge kein Vielfaches von sampling_step ist, hinten abschneiden
    return np.repeat(sampled, sampling_step)[: signal.size]


sampled_signal_hold = sample_and_hold(signal, sampling_frequency, subsampling_frequency)

fig = go.Figure()
fig.add_trace(go.Scatter(x=time, y=sampled_signal_hold, name="Sample & Hold"))
fig.add_trace(
    go.Scatter(
        x=time, y=signal, name="Original", line=dict(color="rgba(255, 0, 0, 0.25)")
    )
)
fig.update_layout(
    xaxis=dict(title="Zeit", ticksuffix=" s", range=[0, 0.5]),
    yaxis=dict(title="Amplitude"),
)
fig.show()

## Alle Ergebnisse in einer Figure

Einmal interaktiv mit Plotly und einmal mit matplotlib.
`sample` liefert einzelne Punkte, `sample_and_hold` eine Treppenfunktion.

In [ ]:
titles = ("a) Signal", "b) sample", "c) sample_and_hold")
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, subplot_titles=titles, vertical_spacing=0.08
)
fig.add_trace(go.Scatter(x=time, y=signal, name="Original"), row=1, col=1)
fig.add_trace(
    go.Scatter(
        x=time,
        y=signal,
        name="Original",
        line=dict(color="rgba(255, 0, 0, 0.25)"),
        showlegend=False,
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(x=sampled_time, y=sampled_signal, name="sample", mode="markers"),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=time,
        y=signal,
        name="Original",
        line=dict(color="rgba(255, 0, 0, 0.25)"),
        showlegend=False,
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scatter(x=time, y=sampled_signal_hold, name="sample_and_hold"), row=3, col=1
)
fig.update_xaxes(range=[0, 0.5])
fig.update_xaxes(title="Zeit", ticksuffix=" s", row=3, col=1)
fig.update_layout(height=800)
fig.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True, layout="constrained")
axes[0].plot(time, signal)
axes[1].plot(time, signal, color="red", alpha=0.25)
axes[1].plot(sampled_time, sampled_signal, ".", label="sample")
axes[2].plot(time, signal, color="red", alpha=0.25)
axes[2].plot(time, sampled_signal_hold, label="sample_and_hold")
for ax, title in zip(
    axes, ("a) Signal", "b) sample", "c) sample_and_hold"), strict=True
):
    ax.set(title=title, ylabel="Amplitude")
axes[2].set(xlabel="Zeit in s", xlim=(0, 0.5))
plt.show()